#Install preqs

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import requests
import re
from torch.utils.data import Dataset, DataLoader
import random
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


#BF4

In [2]:
url = "https://www.gutenberg.org/cache/epub/84/pg84.txt"
print("Downloading Frankenstein from Project Gutenberg...")

response = requests.get(url)
raw_text = response.text

start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK FRANKENSTEIN ***"
end_marker   = "*** END OF THE PROJECT GUTENBERG EBOOK FRANKENSTEIN ***"

try:
    start_idx = raw_text.index(start_marker) + len(start_marker)
    end_idx   = raw_text.index(end_marker)
    text = raw_text[start_idx:end_idx].strip()
except ValueError:
    text = raw_text

text = re.sub(r'\s+', ' ', text)
text = text.replace('\r', '').replace('\n', ' ')
text = text.strip()

print(f"Text length: {len(text):,} characters")
print("First 200 chars:\n", text[:200])

Text length: 437,423 characters
First 200 chars:
 ﻿The Project Gutenberg eBook of Frankenstein; or, the modern prometheus This ebook is for the use of anyone anywhere in the United States and most other parts of the world at no cost and with almost n


#Prep

In [3]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

print(f"Vocabulary size: {vocab_size}")

window_size = 100

inputs = []
targets = []

for i in range(len(text) - window_size):
    seq_in  = text[i : i + window_size - 1]      # 99 chars
    seq_out = text[i + window_size - 1]          # next char (position i+99)

    inputs.append([char_to_idx[c] for c in seq_in])
    targets.append(char_to_idx[seq_out])

print(f"Number of sequences: {len(inputs):,}")

inputs  = torch.tensor(inputs,  dtype=torch.long)
targets = torch.tensor(targets, dtype=torch.long)

Vocabulary size: 92
Number of sequences: 437,323


#Model

In [4]:
class CharRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        embed = self.embedding(x)                # (B, seq_len) → (B, seq_len, embed_dim)
        out, hidden = self.rnn(embed, hidden)    # out: (B, seq_len, hidden_dim)
        out = self.fc(out[:, -1, :])             # take last time step → (B, vocab_size)
        return out, hidden

    def init_hidden(self, batch_size):
        return torch.zeros(1, batch_size, self.rnn.hidden_size).to(device)

# Hyperparameters
embed_dim  = 128
hidden_dim = 256
num_layers = 2

model = CharRNN(vocab_size, embed_dim, hidden_dim, num_layers).to(device)
print(model)

CharRNN(
  (embedding): Embedding(92, 128)
  (rnn): RNN(128, 256, num_layers=2, batch_first=True)
  (fc): Linear(in_features=256, out_features=92, bias=True)
)


In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

batch_size = 128
epochs = 30

dataset = torch.utils.data.TensorDataset(inputs, targets)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

print("Starting training...\n")

for epoch in range(epochs):
    model.train()
    total_loss = 0
    start_time = time.time()

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        output, _ = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Time: {time.time()-start_time:.1f}s")

torch.save(model.state_dict(), "char_rnn_frankenstein.pth")
print("Model saved.")

Starting training...

Epoch 1/30 | Loss: 1.7128 | Time: 34.1s
Epoch 2/30 | Loss: 1.4353 | Time: 35.2s
Epoch 3/30 | Loss: 1.3651 | Time: 34.7s
Epoch 4/30 | Loss: 1.3292 | Time: 34.6s
Epoch 5/30 | Loss: 1.3051 | Time: 34.7s
Epoch 6/30 | Loss: 1.2922 | Time: 34.6s
Epoch 7/30 | Loss: 1.2805 | Time: 34.6s
Epoch 8/30 | Loss: 1.2721 | Time: 34.7s
Epoch 9/30 | Loss: 1.2667 | Time: 34.7s
Epoch 10/30 | Loss: 1.2645 | Time: 34.7s
Epoch 11/30 | Loss: 1.2621 | Time: 34.7s
Epoch 12/30 | Loss: 1.2602 | Time: 34.7s
Epoch 13/30 | Loss: 1.2609 | Time: 34.7s
Epoch 14/30 | Loss: 1.2604 | Time: 34.7s
Epoch 15/30 | Loss: 1.2589 | Time: 34.7s
Epoch 16/30 | Loss: 1.2614 | Time: 34.7s
Epoch 17/30 | Loss: 1.2628 | Time: 34.7s
Epoch 18/30 | Loss: 1.2674 | Time: 34.7s
Epoch 19/30 | Loss: 1.2664 | Time: 34.6s
Epoch 20/30 | Loss: 1.2702 | Time: 34.6s
Epoch 21/30 | Loss: 1.2752 | Time: 34.6s
Epoch 22/30 | Loss: 1.2777 | Time: 34.6s
Epoch 23/30 | Loss: 1.2795 | Time: 34.5s
Epoch 24/30 | Loss: 1.2808 | Time: 34.5s
Epo

In [6]:
def generate_text(model, seed_text, length=200, temperature=0.8):
    model.eval()
    generated = seed_text
    current_seq = [char_to_idx.get(c, 0) for c in seed_text[- (window_size-1): ]]

    with torch.no_grad():
        for _ in range(length):
            x = torch.tensor([current_seq], dtype=torch.long).to(device)  # (1, seq_len)
            out, _ = model(x)

            # Sample from softmax with temperature
            probs = nn.functional.softmax(out / temperature, dim=-1).squeeze().cpu().numpy()
            next_idx = np.random.choice(len(probs), p=probs)

            next_char = idx_to_char[next_idx]
            generated += next_char

            # Slide window
            current_seq = current_seq[1:] + [next_idx]

    return generated

In [7]:
seed = "It was on a dreary night of November"
print("\nGenerated text (temperature 0.8):\n")
print(generate_text(model, seed, length=300, temperature=0.8))

print("\n--- Lower temperature (more deterministic) ---")
print(generate_text(model, seed, length=300, temperature=0.5))

# Cell 8: Try different seeds / temperatures
print("\nAnother sample:")
print(generate_text(model, "The creature", length=250, temperature=0.7))


Generated text (temperature 0.8):

It was on a dreary night of November 19th, 18th; if I cannot and alperiol to a relief my journey in its companion, and if there was good of I was any liken. I. It is the same some restored to his fain would nothing of his sun the ireasant of the perceive, and countenance of the slaves, whither presently sant of Henry of my consent to 

--- Lower temperature (more deterministic) ---
It was on a dreary night of November 100 they which I had followed me. The monster the crress, as the last to really that I am not quickly was more meet the monster, and the same endeavoured the curiose and persips the for my considerable expression of the mortable the same such sink and considerable morning the discovered the waver o

Another sample:
The creature was such as the present in the langth the gloomy that I gated expected until I might religence of the most chosed up to the lakenement of the first grathed the most proceeded me in each expect me quitted some mus